# HCMAN reviewer training on the synthetic kit

Practice for data scientists and cardiologists **before** workspace access. Every record and echo clip here is
synthetic: no patient data, so this runs on a laptop in JupyterLab, never inside the Mayo Clinic Platform workspace.
The screens are the same ones used in the workspace.

| Part | Who | What |
| --- | --- | --- |
| A | Run owner | Build the kit |
| B | Clinical lead | Check the answer key (optional overrides) |
| C | Each trainee | Practice with feedback |
| D | Data scientists, then cardiologists | Mock two-tier audit, with the automatic audit |
| E | Each trainee | Qualification, no feedback |
| F | Run owner | Score |

Setup once: `pip install -r requirements.txt` in the `ml/` folder, then start JupyterLab from `ml/notebooks/`.
Limit: the echo clips are cartoons. They teach the screen and how to mark end-diastole and end-systole, not what
real views look like.

In [ ]:
import sys; sys.path.insert(0, "..")          # the ml/ folder
from pathlib import Path
import pandas as pd
from mcp_adapter.config import StudyConfig
from annotate.store import AnnotationStore
from annotate.synthetic_cases import build_kit, sizes_from_config
from annotate.training import (TrainingScreen, load_training_sets, qualification_report, pair_kappa,
                               refresher_items, refresher_report, marks_from_config)

cfg = StudyConfig.load("../config/study.yaml")
KIT = Path("../training_kit")                 # any local folder
LEAD = cfg.training.get("lead")               # optional: the clinical lead's name

## A. Run owner: build the kit (once)

In [ ]:
if not (KIT / "sets.csv").exists():
    build_kit(KIT, seed=0, **sizes_from_config(cfg))
sets = load_training_sets(KIT / "sets.csv")
STORE = AnnotationStore(KIT / "training.csv")
CTX, CACHE = KIT / "context.pkl", KIT / "cache"
sets.groupby(["set", "task"]).size().unstack(0)

## B. Clinical lead: check the answer key

Every generated answer comes with a teaching note. Work through the screen; any answer you give replaces the
generated one for every trainee (set `training.lead` in the config to your name first).

In [ ]:
TrainingScreen(sets, STORE, LEAD, "reference", phenotype_context=CTX, echo_cache=CACHE).show()

## C. Trainee: practice with feedback

After each answer the screen shows the key's answer and why. Work the whole practice set.

In [ ]:
ME = "<your name>"
TrainingScreen(sets, STORE, ME, "practice", phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

## D. Mock two-tier audit

The mock audit set carries the automatic (code-based) labels, some of them wrong. It runs exactly as in the
workspace: the automatic audit in triage mode, data scientists on tier 1, the cardiologist blinded on tier 2.
Each pair of trainees uses its own audit file.

In [ ]:
from annotate.audit import ECHO_TASK, LabelAudit
from annotate.audit_ui import AuditScreen
from annotate.auto_audit import audit_echo, audit_phenotypes

items = pd.read_csv(KIT / "audit_items.csv", dtype={"item_id": str})
DS, CARD = ["<ds name>"], ["<cardiologist name>"]
mock = LabelAudit(items, AnnotationStore(KIT / f"mock_audit_{DS[0]}.csv"), DS, CARD,
                  spot_check_min=5, auto_tier1="triage")
if mock.auto_verdicts().empty:
    mock.submit_auto(pd.concat([audit_phenotypes(items, CTX),
                                audit_echo(items[items.task == ECHO_TASK], AnnotationStore(KIT / "echo_auto.csv"), CACHE)]))
AuditScreen(mock, DS[0], tier=1, phenotype_context=CTX, echo_cache=CACHE).show()

In [ ]:
# the cardiologist, after tier 1 is finished
AuditScreen(mock, CARD[0], tier=2, phenotype_context=CTX, echo_cache=CACHE).show()

In [ ]:
# debrief: how the audit did against the answer key
from annotate.training import reference_key
key = reference_key(STORE, LEAD)
f = mock.final_labels()
f["key"] = [key[(i, t)][0] for i, t in zip(f.item_id, f.task)]
f["correct"] = [str(a).lower() == str(b).lower() if t != ECHO_TASK else None for a, b, t in zip(f.final_value, f.key, f.task)]
display(mock.report(min_cell=1))
f[f.task != ECHO_TASK].groupby("status").correct.mean()

## E. Trainee: qualification

Attempt 1 uses half A of the reference set, with no feedback. A trainee who misses a mark reviews their errors
with the lead, then takes attempt 2 (half B) once.

In [ ]:
TrainingScreen(sets, STORE, ME, "qualification", attempt=1, phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

## F. Run owner: score

In [ ]:
marks = marks_from_config(cfg)
trainees = {"<ds name 1>": "data_scientist", "<ds name 2>": "data_scientist", "<cardiologist name>": "cardiologist"}
rep = qualification_report(STORE, sets, trainees, attempt=1, marks=marks, lead=LEAD)
rep.to_csv(KIT / "qualification_attempt1.csv", index=False)
rep

In [ ]:
# only when two cardiologists adjudicate
pair = pair_kappa(STORE, sets, "<cardiologist 1>", "<cardiologist 2>", attempt=1)
pair, pair["kappa"] >= marks["cardiologist_pair_kappa"]

Only reviewers with `passed = True` are listed under `audit:` in the config and get workspace accounts.

**Monthly refresher.** During the live audit, each reviewer labels 10 kit items a month, offline.

In [ ]:
MONTH = 1
TrainingScreen(sets, STORE, ME, "refresher", items=refresher_items(sets, MONTH), month=MONTH,
               phenotype_context=CTX, echo_cache=CACHE, lead=LEAD).show()

In [ ]:
refresher_report(STORE, sets, MONTH, list(trainees), lead=LEAD)